In [10]:
# -*- coding: utf-8 -*-
"""YOLO_ViT_Integration.ipynb

YOLO로 객체 탐지 후 Vision Transformer로 분류 수행
"""

import torch
from PIL import Image

from ultralytics import YOLO

import os
import cv2
import numpy as np

In [11]:
# 1. 설정 (Configuration) — .pt 자동 검색 포함
from pathlib import Path

project_root = Path(r"C:\Users\ppos7\Desktop\deep_learning_class_cv-main")
# 자동으로 .pt 파일 검색 (최신 수정된 파일 우선)
pt_files = sorted(project_root.rglob("*.pt"), key=lambda p: p.stat().st_mtime, reverse=True)

if pt_files:
    YOLO_MODEL_PATH = str(pt_files[0])
    print("→ 자동으로 찾은 YOLO 가중치:", YOLO_MODEL_PATH)
else:
    print("✗ .pt 파일을 찾지 못했습니다. 터미널에서 수동 검색을 수행하거나 YOLO_MODEL_PATH를 직접 설정하세요.")
    print(r"  CMD:  dir /s /b *.pt")
    print(r"  PowerShell: Get-ChildItem -Path . -Recurse -Filter *.pt")
    raise FileNotFoundError(".pt 파일을 찾을 수 없습니다. YOLO_MODEL_PATH를 설정하세요.")

# ViT/이미지 경로는 기존 값 유지
VIT_MODEL_PATH = r"C:\Users\ppos7\Desktop\deep_learning_class_cv-main\Vision_transformer_CIFAR10\ViT_inference.py"
IMAGE_PATH = r"C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\custom_images\cans\cans (1).jpg"
YOLO_CONF_THRESHOLD = 0.15
RESULT_SAVE_NAME = 'yolo_vit_result'

→ 자동으로 찾은 YOLO 가중치: C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\weights\yolov10n.pt


In [ ]:
# 2. ViT 모델 로드 (수정)
# ========================================
import importlib.util
import torch  # <- 명시적 import 추가 (NameError 방지)
print("=" * 50)
print("ViT 모델 로딩 중...")
print("=" * 50)

try:
    from transformers import ViTImageProcessor, ViTForImageClassification
except Exception as e:
    print("✗ transformers 설치 필요:", e)
    print("터미널에서: pip install transformers")
    raise

try:
    # 설정 셀에서 VIT_MODEL_PATH가 정의되지 않았을 수 있으므로 체크
    if 'VIT_MODEL_PATH' not in globals():
        raise RuntimeError("VIT_MODEL_PATH가 정의되어 있지 않습니다. 설정(1) 셀을 먼저 실행하세요.")

    # VIT_MODEL_PATH가 .py 파일이면 로컬 스크립트에서 모델 생성 함수를 찾기 시도
    if VIT_MODEL_PATH.lower().endswith(".py"):
        spec = importlib.util.spec_from_file_location("vit_local", VIT_MODEL_PATH)
        vit_local = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(vit_local)

        # 로컬 스크립트에 모델/프로세서 반환 함수가 있다고 가정
        if hasattr(vit_local, "load_model"):
            vit_model, vit_processor = vit_local.load_model()
        elif hasattr(vit_local, "get_model"):
            vit_model, vit_processor = vit_local.get_model()
        else:
            raise RuntimeError("ViT 스크립트에 load_model/get_model 함수가 없습니다. 로컬 스크립트 대신 모델 디렉터리(.json 포함) 또는 HF 모델 id를 사용하세요.")
    else:
        # Hugging Face 디렉터리 또는 모델 id로부터 로드 (config.json 필요)
        vit_processor = ViTImageProcessor.from_pretrained(VIT_MODEL_PATH)
        vit_model = ViTForImageClassification.from_pretrained(VIT_MODEL_PATH)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vit_model.to(device)
    vit_model.eval()
    print(f"ViT 모델 로드 완료. Device: {device}")

except Exception as e:
    print(f"ViT 모델 로드 중 오류 발생: {e}")
    print("폴더 경로와 파일 존재 여부를 확인하거나, ViT_inference.py가 모델 반환 함수를 제공하는지 확인하세요.")
    raise


ViT 모델 로딩 중...
✓ ViT 모델 로드 완료. Device: cpu


In [12]:
# 3. YOLO 모델 로드
# ========================================
import os
import glob
from ultralytics import YOLO  # ensure correct class

print("\n" + "=" * 50)
print("YOLO 모델 로딩 중...")
print("=" * 50)

# 명시적 경로가 없으면 자동으로 .pt 파일 검색
def find_yolo_weights(path_hint="./runs/detect", pattern="*.pt"):
    candidates = glob.glob(os.path.join(path_hint, "**", pattern), recursive=True)
    candidates = [c for c in candidates if os.path.isfile(c)]
    return sorted(candidates, key=os.path.getmtime, reverse=True)

# YOLO_MODEL_PATH가 정의되지 않았으면 자동 검색으로 채움
if 'YOLO_MODEL_PATH' not in globals():
    print("→ 설정에 YOLO_MODEL_PATH가 정의되어 있지 않습니다. 자동으로 .pt 파일을 검색합니다.")
    found = find_yolo_weights()
    if found:
        YOLO_MODEL_PATH = found[0]
        print(f"→ 자동으로 찾은 가중치 사용: {YOLO_MODEL_PATH}")
    else:
        print("자동 검색으로도 .pt 파일을 찾지 못했습니다. YOLO_MODEL_PATH를 설정하세요.")

        raise FileNotFoundError("YOLO weights not found and YOLO_MODEL_PATH not set.")

# 설정된 경로 존재 확인
if not os.path.exists(YOLO_MODEL_PATH):
    print(f"지정된 YOLO 모델 파일 '{YOLO_MODEL_PATH}'을 찾을 수 없습니다.")
    # 자동 검색 재시도
    found = find_yolo_weights()
    if found:
        YOLO_MODEL_PATH = found[0]
        print(f"자동으로 찾은 가중치 사용: {YOLO_MODEL_PATH}")
    else:
        print("자동 검색으로도 .pt 파일을 찾지 못했습니다. 경로를 확인하세요.")
        raise FileNotFoundError(f"YOLO weights not found: {YOLO_MODEL_PATH}")

try:
    yolo_model = YOLO(YOLO_MODEL_PATH)
    print(f"YOLO 모델 로드 완료: {YOLO_MODEL_PATH}")
except Exception as e:
    print(f"YOLO 모델 로드 중 오류 발생: {e}")
    print("ultralytics가 설치되어 있고, 모델 파일이 손상되지 않았는지 확인하세요.")
    print("설치/업데이트: pip install --upgrade ultralytics")
    raise


YOLO 모델 로딩 중...
YOLO 모델 로드 완료: C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\weights\yolov10n.pt


In [13]:
# 4. 이미지 로드 및 검증
# ========================================

print("\n" + "=" * 50)
print("이미지 로딩 중...")
print("=" * 50)

if not os.path.exists(IMAGE_PATH):
    print(f"✗ 이미지 파일 '{IMAGE_PATH}'을 찾을 수 없습니다.")
    exit()

try:
    original_image = cv2.imread(IMAGE_PATH)
    original_image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    print(f"이미지 로드 완료: {IMAGE_PATH}")
    print(f"  이미지 크기: {original_image.shape[1]}x{original_image.shape[0]}")
except Exception as e:
    print(f"이미지 로드 중 오류 발생: {e}")
    exit()


이미지 로딩 중...
이미지 로드 완료: C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\custom_images\cans\cans (1).jpg
  이미지 크기: 100x100


In [14]:
# 5. YOLO 객체 탐지 수행
# ========================================

print("\n" + "=" * 50)
print("YOLO 객체 탐지 수행 중...")
print("=" * 50)

yolo_results = yolo_model.predict(
    source=IMAGE_PATH,
    conf=YOLO_CONF_THRESHOLD,
    save=False,
    verbose=False
)

num_detections = len(yolo_results[0].boxes)
print(f"검출된 총 객체 수: {num_detections}")

# Debug: YOLO 예측 상세 출력 및 시각화
print(">> yolo_model:", type(yolo_model))
print(">> IMAGE_PATH:", IMAGE_PATH)
print(">> YOLO_CONF_THRESHOLD:", YOLO_CONF_THRESHOLD)

# 재실행: 더 낮은 임계값, verbose 켜기, 이미지 사이즈 조정 시도
debug_results = yolo_model.predict(source=IMAGE_PATH, conf=0.1, verbose=True, imgsz=640)

print(">> results type:", type(debug_results), "len:", len(debug_results))
r = debug_results[0]
# boxes 정보 출력 (유효하면 .boxes 속성 존재)
if hasattr(r, "boxes"):
    boxes = r.boxes
    try:
        print(">> boxes count (len):", len(boxes))
        # xyxy, conf, cls 출력 (각 속성 존재 확인)
        if hasattr(boxes, "xyxy"):
            print("xyxy:", boxes.xyxy.cpu().numpy())
        if hasattr(boxes, "conf"):
            print("conf:", boxes.conf.cpu().numpy())
        if hasattr(boxes, "cls"):
            print("cls:", boxes.cls.cpu().numpy())
    except Exception as e:
        print(">> boxes 속성 읽기 오류:", e)
else:
    print(">> 결과에 boxes 속성이 없습니다. raw result:", r)

# 시각화: 플롯된 이미지 저장/표시
try:
    annotated = r.plot()  # numpy BGR 또는 RGB depending on ultralytics
    from PIL import Image
    im = Image.fromarray(annotated)
    save_path = "yolo_debug_annotated.jpg"
    im.save(save_path)
    print("→ 어노테이션 이미지 저장:", save_path)
except Exception as e:
    print("→ 시각화 실패:", e)


YOLO 객체 탐지 수행 중...
검출된 총 객체 수: 1
>> yolo_model: <class 'ultralytics.models.yolo.model.YOLO'>
>> IMAGE_PATH: C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\custom_images\cans\cans (1).jpg
>> YOLO_CONF_THRESHOLD: 0.15

image 1/1 C:\Users\ppos7\Desktop\deep_learning_class_cv-main\YOLOv10\custom_images\cans\cans (1).jpg: 640x640 1 cup, 1 toothbrush, 50.5ms
Speed: 2.3ms preprocess, 50.5ms inference, 0.2ms postprocess per image at shape (1, 3, 640, 640)
>> results type: <class 'list'> len: 1
>> boxes count (len): 2
xyxy: [[     34.874      24.114      61.598      89.606]
 [     25.644      5.9876      71.893      99.125]]
conf: [    0.22095     0.14462]
cls: [         79          41]
→ 어노테이션 이미지 저장: yolo_debug_annotated.jpg


In [16]:
# 7. 결과 저장
# ========================================

print("\n" + "=" * 50)
print("결과 저장 중...")
print("=" * 50)

save_dir = f'./runs/detect/{RESULT_SAVE_NAME}'
os.makedirs(save_dir, exist_ok=True)

result_image_path = os.path.join(save_dir, 'result.jpg')
cv2.imwrite(result_image_path, results_image)
print(f"결과 이미지 저장 완료: {result_image_path}")

# 결과를 텍스트 파일로도 저장
result_text_path = os.path.join(save_dir, 'results.txt')
with open(result_text_path, 'w', encoding='utf-8') as f:
    f.write(f"총 검출 객체 수: {num_detections}\n\n")
    for idx, result in enumerate(classification_results):
        f.write(f"객체 #{idx + 1}\n")
        f.write(f"  위치: {result['bbox']}\n")
        f.write(f"  YOLO 탐지: {result['yolo_class']} (신뢰도: {result['yolo_conf']:.3f})\n")
        f.write(f"  ViT 분류: {result['vit_class']} (신뢰도: {result['vit_conf']:.3f})\n\n")

print(f"결과 텍스트 저장 완료: {result_text_path}")

print("\n" + "=" * 50)
print("모든 처리 완료!")
print("=" * 50)


결과 저장 중...
결과 이미지 저장 완료: ./runs/detect/yolo_vit_result\result.jpg
결과 텍스트 저장 완료: ./runs/detect/yolo_vit_result\results.txt

모든 처리 완료!
